# **IF 800 - Final Project**

Nama: Wilbert Bernardi

NIM: 00000069185

Angkatan: 2022

Judul: Analisis Sentimen Terhadap Permasalahan Konflik Rusia dan Ukraina

In [ ]:
from googleapiclient.discovery import build

api_key = "YOUR_API_KEY"  # Replace with your actual YouTube Data API key
youtube = build('youtube', 'v3', developerKey=api_key)

def get_comments(video_id):
    comments = []

    request = youtube.commentThreads().list(
        part="snippet",
        videoId=video_id,
        maxResults=100
    )

    while request:
        response = request.execute()

        for item in response['items']:
            comment_snippet = item['snippet']['topLevelComment']['snippet']
            comment_data = {
                'textDisplay': comment_snippet['textDisplay'],
                'authorDisplayName': comment_snippet['authorDisplayName'],
                'likeCount': comment_snippet['likeCount'],
                'publishedAt': comment_snippet['publishedAt']
            }
            comments.append(comment_data)

        request = youtube.commentThreads().list_next(request, response)

    return comments

video1 = "wsa8w2CSD1A"
video2 = "9PX1kgbuTEw"
video3 = "tCFiKNdcNLg"
video4 = "UV-FeawizZg"
video5 = "E-tnuyB40Pw"
video6 = "67w4lg3jnEY"

comments1 = get_comments(video1)
comments2 = get_comments(video2)
comments3 = get_comments(video3)
comments4 = get_comments(video4)
comments5 = get_comments(video5)
comments6 = get_comments(video6)

In [ ]:
comments1

In [ ]:
comments2

In [ ]:
comments3

In [ ]:
comments4

In [ ]:
comments5

In [ ]:
comments6

In [ ]:
import pandas as pd

# Combine comments from both videos
all_comments = comments1 + comments2 + comments3 + comments4 + comments5 + comments6

# Create a DataFrame from the combined comments
df_comments = pd.DataFrame(all_comments)

# Rename 'textDisplay' column to 'comment_text' for consistency
df_comments = df_comments.rename(columns={'textDisplay': 'comment_text'})

# Export the DataFrame to a CSV file
csv_filename = 'youtube_comments_rusia_ukraine_conflict.csv'
df_comments.to_csv(csv_filename, index=False, encoding='utf-8')

print(f"All comments have been successfully exported to '{csv_filename}'")
print(f"Number of comments exported: {len(df_comments)}")

#**Import Dataset & Analyze Data**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Dataset/youtube_comments_rusia_ukraine_conflict.csv')

df

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.shape

#**Menginstalasi library stopwords**

In [ ]:
pip install nltk

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [ ]:
# Pastikan resource punkt_tab terunduh untuk menghindari LookupError
nltk.download('punkt_tab')

# **Preprocessing**

#### **Case Fold**

In [ ]:
import re
import string

def case_fold_text(text):
    text = str(text).lower()
    return text

# Terapkan fungsi pada dataframe
df['case_fold_text'] = df['comment_text'].apply(case_fold_text)

# Tampilkan perbandingan untuk beberapa contoh
print(f"Data berhasil diproses. Total baris: {len(df)}")
print("\nPerbandingan Case Folding (5 contoh pertama):")
for i in range(5):
    print(f"Before: {df['comment_text'].iloc[i]}")
    print(f"After: {df['case_fold_text'].iloc[i]}")
    print("-" * 30)

In [ ]:
display(df[['comment_text', 'case_fold_text']].head())

#### **Cleaning**

In [ ]:
import re
import string

def cleaning_text(text):
  # Menghapus URL dan HTML tags
  text = re.sub(r'https?://\S+|www\.\S+|<.*?>', '', text)

  # Menghapus emoji
  emoji_pattern = re.compile("["
                           "\U0001F600-\U0001F64F"  # emoticons
                           "\U0001F300-\U0001F5FF"  # symbols & pictographs
                           "\U0001F680-\U0001F6FF"  # transport & map symbols
                           "\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           "\U00002702-\U000027B0" # Dingbats
                           "\U000024C2-\U0001F251" # Enclosed Alphanumerics, etc.
                           "\U0000200D" # Zero Width Joiner
                           "\U0000FE0F" # Variation Selector
                           "\U00002600-\U000026FF" # Miscellaneous Symbols
                           "\U00002100-\U000021FF" # Letterlike Symbols
                           "\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs (includes facepalm)
                           "]+", flags=re.UNICODE)
  text = emoji_pattern.sub(r'', text)

  # Menghapus tanda baca dan angka
  text = text.translate(str.maketrans('', '', string.punctuation + string.digits))

  # Menghapus banyak spasi berlebih menjadi satu spasi
  text = re.sub(r'\s+', ' ', text).strip()

  return text

# Terapkan fungsi pada dataframe
df['cleaning_text'] = df['case_fold_text'].apply(cleaning_text)

# Tampilkan perbandingan untuk beberapa contoh
print(f"Data berhasil diproses. Total baris: {len(df)}")
print("\nPerbandingan Cleaning (5 contoh pertama):")
for i in range(5):
    print(f"Before: {df['case_fold_text'].iloc[i]}")
    print(f"After: {df['cleaning_text'].iloc[i]}")
    print("-" * 30)

In [ ]:
display(df[['case_fold_text', 'cleaning_text']].head())

#### **Normalization**

In [ ]:
# Kamus Normalisasi
kamus_df = pd.read_csv("/content/drive/MyDrive/Dataset/kamus_normalisasi_kata_tidak_baku.csv", encoding="latin1", header=None)
kamus_df.columns = ["tidak_baku", "baku"]

# ubah menjadi dictionary
normalization_dict = dict(zip(kamus_df["tidak_baku"], kamus_df["baku"]))

# Tambahkan slang yang berhubungan dengan topik perang Rusia dan Ukraina
war_slang_dict = {
    "mamarik": "amerika",
    "mamarika": "amerika",
    "uraaa": "hidup rusia", # Interpretasi kontekstual untuk dukungan terhadap Rusia
    "ura": "hidup rusia",
    "badut": "komedian", # Sering merujuk pada Zelensky
    "zelenskt": "zelensky",
    "ukrain": "ukraina",
    "nato": "pakta pertahanan atlantik utara", # Nama lengkap untuk kejelasan
    "as": "amerika serikat",
    "pd3": "perang dunia tiga",
    "zionis": "zionis",
    "rusia": "rusia", # Memastikan konsistensi
    "putin": "putin"
}

normalization_dict.update(war_slang_dict) # Perbarui kamus utama dengan slang terkait perang

def normalize_text(text):
    words = text.split()
    normalized_words = [normalization_dict.get(word, word) for word in words]
    normalized_text = ' '.join(normalized_words)

    return normalized_text

# Terapkan fungsi pada dataframe
df['normalize_text'] = df['cleaning_text'].apply(normalize_text)

# Tampilkan perbandingan untuk beberapa contoh
print(f"Data berhasil diproses. Total baris: {len(df)}")
print("\nPerbandingan Normalization (5 contoh pertama):")
for i in range(5):
    print(f"Before: {df['cleaning_text'].iloc[i]}")
    print(f"After: {df['normalize_text'].iloc[i]}")
    print("-" * 30)

In [ ]:
display(df[['cleaning_text', 'normalize_text']].head())

#### **Tokenize**

In [ ]:
def tokenize_text(text):
    tokens = word_tokenize(text)
    return tokens

# Terapkan fungsi pada dataframe
df['tokenize_text'] = df['normalize_text'].apply(tokenize_text)

# Tampilkan perbandingan untuk beberapa contoh
print(f"Data berhasil diproses. Total baris: {len(df)}")
print("\nPerbandingan Tokenization (5 contoh pertama):")
for i in range(5):
    print(f"Before: {df['normalize_text'].iloc[i]}")
    print(f"After: {df['tokenize_text'].iloc[i]}")
    print("-" * 30)

In [ ]:
display(df[['normalize_text', 'tokenize_text']].head())

#### **Stopwords**

In [ ]:
# Ambil daftar stopword bahasa Indonesia dari NLTK
stop_words = set(stopwords.words('indonesian'))

def remove_stopwords_text(tokens):
    if not isinstance(tokens, list): # Ensure the input is a list of tokens
        return [] # Return empty list if not a list
    filtered_words = [word for word in tokens if word not in stop_words]
    return filtered_words # Return as a list of words

# Terapkan fungsi pada dataframe
df['stopwords_removed_text'] = df['tokenize_text'].apply(remove_stopwords_text)

# Tampilkan perbandingan untuk beberapa contoh
print(f"Data berhasil diproses. Total baris: {len(df)}")
print("\nPerbandingan Stopwords Removal (5 contoh pertama):")
for i in range(5):
    print(f"Before: {df['tokenize_text'].iloc[i]}")
    print(f"After: {df['stopwords_removed_text'].iloc[i]}")
    print("-" * 30)

In [ ]:
display(df[['tokenize_text', 'stopwords_removed_text']].head())

#### **Stemming**

In [ ]:
pip install Sastrawi

In [ ]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Create stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stem_text(tokens):
    if not isinstance(tokens, list) or not tokens: # Check if it's a list and not empty
        return "" # Return empty string for empty/invalid input
    text_to_stem = " ".join(tokens) # Join tokens into a single string
    return stemmer.stem(text_to_stem)

# Apply stemming to the 'stopwords_removed_text' column
df['stemmed_text'] = df['stopwords_removed_text'].apply(stem_text)

print(f"Data berhasil diproses. Total baris: {len(df)}")
print("\nPerbandingan Stemming (5 contoh pertama):")
for i in range(5):
    print(f"Before: {df['stopwords_removed_text'].iloc[i]}")
    print(f"After: {df['stemmed_text'].iloc[i]}")
    print("-" * 30)

In [ ]:
display(df[['stopwords_removed_text', 'stemmed_text']])

#### **Check if it's null or not**

In [ ]:
import numpy as np

# Check for null values
null_count = df['stemmed_text'].isnull().sum()
print(f"Jumlah nilai null di kolom 'stemmed_text': {null_count}")

# Check for empty strings
empty_string_count = (df['stemmed_text'] == '').sum()
print(f"Jumlah string kosong di kolom 'stemmed_text': {empty_string_count}")

# Optionally, check for whitespace-only strings if they should be considered empty
whitespace_only_count = df['stemmed_text'].apply(lambda x: isinstance(x, str) and x.strip() == '').sum()
print(f"Jumlah string hanya spasi di kolom 'stemmed_text': {whitespace_only_count}")

In [ ]:
df_cleaned = df[df['stemmed_text'].astype(bool)]

print(f"Jumlah baris sebelum penghapusan data kosong: {len(df)}")
print(f"Jumlah baris setelah penghapusan data kosong: {len(df_cleaned)}")

df = df_cleaned.copy()

In [ ]:
df_cleaned['stemmed_text']

In [ ]:
df_cleaned['stemmed_text'].info()

# **Labelling**

#### **Lexicon Labeling [Indonesian Sentiment (InSet) Lexicon]**

In [ ]:
# Load positive and negative keyword datasets
positive_keywords_df = pd.read_csv('/content/drive/MyDrive/Dataset/positive.tsv', sep='\t', header=None, names=['keyword', 'weight'])
negative_keywords_df = pd.read_csv('/content/drive/MyDrive/Dataset/negative.tsv', sep='\t', header=None, names=['keyword', 'weight'])

# Convert 'weight' columns to numeric type
positive_keywords_df['weight'] = pd.to_numeric(positive_keywords_df['weight'], errors='coerce').fillna(0).astype(int)
negative_keywords_df['weight'] = pd.to_numeric(negative_keywords_df['weight'], errors='coerce').fillna(0).astype(int)

# Convert keywords to dictionaries for faster lookup, mapping keyword to its weight
positive_keywords = dict(zip(positive_keywords_df['keyword'].str.lower(), positive_keywords_df['weight']))
negative_keywords = dict(zip(negative_keywords_df['keyword'].str.lower(), negative_keywords_df['weight']))

def classify_sentiment(text):
    if not isinstance(text, str) or not text.strip():  # Handle non-string or empty inputs
        return 'Neutral'

    words = text.lower().split()
    positive_score = sum(positive_keywords.get(word, 0) for word in words)
    negative_score = sum(negative_keywords.get(word, 0) for word in words)

    # Calculate the net polarization score
    net_score = positive_score + negative_score

    if net_score > 0:
        return 'Positive'
    elif net_score < 0:
        return 'Negative'
    else:
        return 'Neutral'

# Apply the sentiment classification function to the 'stemmed_text' column
df_cleaned.loc[:, 'sentiment_label'] = df_cleaned['stemmed_text'].apply(classify_sentiment)

# Display some labeled comments
print("Comments with Sentiment Labels:\n")
display(df_cleaned[['comment_text', 'stemmed_text', 'sentiment_label']].sample(10))

# Display the distribution of sentiment labels with percentages
print("\nSentiment Label Distribution:")
sentiment_counts = df_cleaned['sentiment_label'].value_counts()
sentiment_percentages = df_cleaned['sentiment_label'].value_counts(normalize=True) * 100

for label in sentiment_counts.index:
    count = sentiment_counts[label]
    percentage = sentiment_percentages[label]
    print(f"{label}    {count} ({percentage:.2f}%)")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate sentiment percentages
sentiment_percentages = df_cleaned['sentiment_label'].value_counts(normalize=True) * 100

# Create a DataFrame for plotting
sentiment_df = pd.DataFrame({
    'Sentiment': sentiment_percentages.index,
    'Percentage': sentiment_percentages.values
})

# Define a custom color palette for sentiment labels (e.g., Negative: red, Neutral: blue, Positive: green)
sentiment_order = ['Negative', 'Neutral', 'Positive']
colors = []
for sentiment in sentiment_order:
    if sentiment == 'Negative':
        colors.append('red')
    elif sentiment == 'Neutral':
        colors.append('blue')
    elif sentiment == 'Positive':
        colors.append('green')

plt.figure(figsize=(8, 6))
sns.barplot(x='Sentiment', y='Percentage', data=sentiment_df, palette=colors, order=sentiment_order, hue='Sentiment', legend=False)
plt.title('Sentiment Distribution of Comments')
plt.xlabel('Sentiment')
plt.ylabel('Percentage')
plt.ylim(0, 100)

# Prepare sentiment_df to match the order of the bars for correct label placement
sentiment_df_ordered = sentiment_df.set_index('Sentiment').reindex(sentiment_order).reset_index()

# Add percentage labels on top of the bars
for index, row in sentiment_df_ordered.iterrows():
    plt.text(index, row.Percentage + 1, f'{row.Percentage:.2f}%', color='black', ha="center")

plt.show()

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Create a dictionary to store concatenated stemmed text for each sentiment
sentiment_texts = {
    'Positive': ' ',
    'Negative': ' ',
    'Neutral': ' '
}

# Populate the dictionary with stemmed text based on sentiment_label
for index, row in df_cleaned.iterrows():
    label = row['sentiment_label']
    text = row['stemmed_text']
    if label in sentiment_texts:
        sentiment_texts[label] += text + ' '

# Generate and display word cloud for Positive sentiment only
plt.figure(figsize=(10, 8))

positive_text = sentiment_texts['Positive']
if positive_text.strip():
    wordcloud = WordCloud(width=800, height=400, background_color='white', colormap='Blues').generate(positive_text)
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.title('Word Cloud for Positive Sentiment')
    plt.axis('off')
else:
    plt.text(0.5, 0.5, 'No text for Positive sentiment', horizontalalignment='center', verticalalignment='center')
    plt.title('Word Cloud for Positive Sentiment')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))

negative_text = sentiment_texts['Negative']
if negative_text.strip():
    wordcloud = WordCloud(width=800, height=400, background_color='white', colormap='Reds').generate(negative_text)
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.title('Word Cloud for Negative Sentiment')
    plt.axis('off')
else:
    plt.text(0.5, 0.5, 'No text for Negative sentiment', horizontalalignment='center', verticalalignment='center')
    plt.title('Word Cloud for Negative Sentiment')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))

neutral_text = sentiment_texts['Neutral']
if neutral_text.strip():
    wordcloud = WordCloud(width=800, height=400, background_color='white', colormap='Greens').generate(neutral_text)
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.title('Word Cloud for Neutral Sentiment')
    plt.axis('off')
else:
    plt.text(0.5, 0.5, 'No text for Neutral sentiment', horizontalalignment='center', verticalalignment='center')
    plt.title('Word Cloud for Neutral Sentiment')
    plt.axis('off')

plt.tight_layout()
plt.show()

# **Split Dataset**

In [ ]:
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
X = df_cleaned['stemmed_text']
y = df_cleaned['sentiment_label']

# First split: 70% training, 30% temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Second split: temp into 50% validation and 50% test (which is 15% of original data each)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Original dataset size: {len(df_cleaned)}")
print(f"Training set size: {len(X_train)} (comments) / {len(y_train)} (labels)")
print(f"Validation set size: {len(X_val)} (comments) / {len(y_val)} (labels)")
print(f"Testing set size: {len(X_test)} (comments) / {len(y_test)} (labels)")

In [ ]:
import pandas as pd

# --- Display Lexicon-based Sentiment Label Distribution for Split Data ---
# The y_train and y_test variables already contain the lexicon-based sentiment labels
# derived from the 'sentiment_label' column of df_cleaned, which was classified using the lexicon.

print("\nLexicon Sentiment Label Distribution (Training Set):")
print(y_train.value_counts())
print("\nLexicon Sentiment Label Distribution (Validation Set):")
print(y_val.value_counts())
print("\nLexicon Sentiment Label Distribution (Testing Set):")
print(y_test.value_counts())

# **Feature Extraction (Embedding)**

In [ ]:
import torch
from transformers import BertTokenizer, AutoModel
from google.colab import userdata
import pandas as pd

# Load your Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get('HF_TOKEN')

# Load pre-trained IndoBERT tokenizer and model
# 'indobert-base-p1' is a good choice for Indonesian text
model_name = "indobenchmark/indobert-base-p1"
tokenizer = BertTokenizer.from_pretrained(model_name, token=HF_TOKEN)
model = AutoModel.from_pretrained(model_name, token=HF_TOKEN)

# Function to generate BERT embeddings
def get_bert_embeddings(text):
    if not isinstance(text, str) or not text.strip():
        # Return a tensor of zeros for empty or non-string inputs
        # The size should match the BERT model's hidden size (768 for base models)
        return torch.zeros(model.config.hidden_size).tolist()

    # Tokenize the text, explicitly setting max_length
    encoded_input = tokenizer(text, padding='max_length', truncation=True, max_length=128, return_tensors='pt')

    # Get model outputs
    with torch.no_grad():
        model_output = model(**encoded_input)

    # Use the [CLS] token embedding as the sentence embedding (first token)
    sentence_embedding = model_output.last_hidden_state[:, 0, :].squeeze()

    return sentence_embedding.tolist()

# Apply the function directly to the split data (X_train, X_val, X_test)
X_train_embeddings = pd.DataFrame(list(X_train.apply(get_bert_embeddings)), index=X_train.index)
X_val_embeddings = pd.DataFrame(list(X_val.apply(get_bert_embeddings)), index=X_val.index)
X_test_embeddings = pd.DataFrame(list(X_test.apply(get_bert_embeddings)), index=X_test.index)

print("BERT embeddings generated successfully for X_train, X_val, and X_test!")

In [ ]:
# Display shape of train, validation, test split data after embedding process

print(f"Shape of X_train_embeddings: {X_train_embeddings.shape}")
print(f"Shape of X_val_embeddings: {X_val_embeddings.shape}")
print(f"Shape of X_test_embeddings: {X_test_embeddings.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_val: {y_val.shape}")
print(f"Shape of y_test: {y_test.shape}")

In [ ]:
print("\n--- Sample of X_train_embeddings paired with y_train ---")
# Display the first 5 rows of training embeddings and their corresponding labels
# Ensure indices match

sampled_indices = X_train.head(5).index
sample_embeddings = X_train_embeddings.loc[sampled_indices]
sample_labels = y_train.loc[sampled_indices]

# Create a DataFrame for better display
display_df = pd.DataFrame({
    'Embedding Sample (first 5 dimensions)': sample_embeddings.iloc[:, :5].values.tolist(),
    'Original Label': sample_labels.values
})
display(display_df)

# **Balancing**

In [ ]:
from imblearn.over_sampling import RandomOverSampler

# Initialize RandomOverSampler
ros = RandomOverSampler(random_state=42)

# Resample the training data (embeddings)
X_train_resampled_embeddings, y_train_resampled = ros.fit_resample(X_train_embeddings, y_train)

# Display the new distribution of sentiment labels in the resampled training set
print("Distribution of sentiment labels after oversampling:")
print(y_train_resampled.value_counts())

# You can also check the percentage distribution
print("\nPercentage distribution of sentiment labels after oversampling:")
print(y_train_resampled.value_counts(normalize=True) * 100)

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

# Initialize RandomUnderSampler
rus = RandomUnderSampler(random_state=42)

# Resample the training data (embeddings) using undersampling
X_train_undersampled_embeddings, y_train_undersampled = rus.fit_resample(X_train_embeddings, y_train)

# Display the new distribution of sentiment labels in the undersampled training set
print("Distribution of sentiment labels after undersampling:")
print(y_train_undersampled.value_counts())

# You can also check the percentage distribution
print("\nPercentage distribution of sentiment labels after undersampling:")
print(y_train_undersampled.value_counts(normalize=True) * 100)

# **Fine-Tuning**

## **Batch Size 16 Learning rate 2e-5**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

# 1. Encode labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train_embeddings.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_encoded, dtype=torch.long)
X_val_tensor = torch.tensor(X_val_embeddings.values, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_encoded, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_embeddings.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_encoded, dtype=torch.long)

In [ ]:
# 2. Define the Neural Network Classifier
class BertClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dim=256):
        super(BertClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim) # Dense layer 1
        self.relu = nn.ReLU()                       # Activation function
        self.dropout = nn.Dropout(0.3)              # Dropout for regularization
        self.fc2 = nn.Linear(hidden_dim, num_classes) # Dense layer 2 (output layer)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        # Softmax is typically applied by CrossEntropyLoss, so we return logits
        return x

# Instantiate the model
input_dim = X_train_embeddings.shape[1] # 768 for IndoBERT base embeddings
num_classes = len(label_encoder.classes_) # 3 (Negative, Neutral, Positive)
model = BertClassifier(input_dim, num_classes)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00002)

# Create DataLoader
batch_size = 16
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# 3. Training loop with Validation
num_epochs = 10
print("Starting fine-tuning with custom dense layers...")

for epoch in range(num_epochs):
    model.train() # Set model to training mode
    total_train_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad() # Clear gradients
        outputs = model(batch_X) # Forward pass
        loss = criterion(outputs, batch_y) # Compute loss
        loss.backward() # Backward pass
        optimizer.step() # Update weights
        total_train_loss += loss.item()

    # Validation step
    model.eval() # Set model to evaluation mode
    total_val_loss = 0
    val_correct = 0
    total_val_samples = 0
    with torch.no_grad(): # Disable gradient calculation during evaluation
        for batch_X_val, batch_y_val in val_loader:
            outputs_val = model(batch_X_val)
            loss_val = criterion(outputs_val, batch_y_val)
            total_val_loss += loss_val.item()
            _, predicted_val = torch.max(outputs_val.data, 1) # Get the class with the highest probability
            total_val_samples += batch_y_val.size(0)
            val_correct += (predicted_val == batch_y_val).sum().item()

    val_accuracy = val_correct / total_val_samples

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {total_train_loss/len(train_loader):.4f}, Val Loss: {total_val_loss/len(val_loader):.4f}, Val Accuracy: {val_accuracy:.4f}")

print("Fine-tuning complete.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Data extracted from the output of the previous training cell (UrxHdBX5SSat)
epochs = range(1, 11)
train_losses = [0.8491, 0.7694, 0.7451, 0.7233, 0.7066, 0.7006, 0.6894, 0.6729, 0.6651, 0.6598]
val_losses = [0.7828, 0.7558, 0.7373, 0.7243, 0.7102, 0.7022, 0.6944, 0.6885, 0.6811, 0.6766]
val_accuracies = [0.6737, 0.6863, 0.7045, 0.7101, 0.7115, 0.7185, 0.7269, 0.7255, 0.7227, 0.7241]

plt.figure(figsize=(12, 5))

# Plotting Loss
plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
plt.plot(epochs, train_losses, label='Training Loss')
plt.plot(epochs, val_losses, label='Validation Loss')
plt.title('Training and Validation Loss per Epoch (Baseline(BS 16, LR 2e-5))')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plotting Accuracy
plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
plt.plot(epochs, val_accuracies, label='Validation Accuracy', color='green')
plt.title('Validation Accuracy per Epoch (Baseline(BS 16, LR 2e-5))')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# 4. Evaluation
model.eval() # Set model to evaluation mode
all_preds = []
all_labels = []

with torch.no_grad(): # Disable gradient calculation during evaluation
    for batch_X, batch_y in test_loader:
        outputs = model(batch_X)
        _, predicted = torch.max(outputs.data, 1) # Get the class with the highest probability
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())

# Convert numerical predictions back to original labels for classification report
y_pred_fine_tuned = label_encoder.inverse_transform(all_preds)
y_test_original = label_encoder.inverse_transform(all_labels)

print("\nEvaluation Results (Baseline (BS 16, LR 2e-5)):")
print(f"Accuracy: {accuracy_score(y_test_original, y_pred_fine_tuned):.4f}")

# Calculate and print additional metrics explicitly
from sklearn.metrics import precision_score, recall_score, f1_score

print(f"Precision: {precision_score(y_test_original, y_pred_fine_tuned, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test_original, y_pred_fine_tuned, average='weighted'):.4f}")
print(f"F1-Score: {f1_score(y_test_original, y_pred_fine_tuned, average='weighted'):.4f}")

print("\nClassification Report:")
print(classification_report(y_test_original, y_pred_fine_tuned, target_names=label_encoder.classes_, digits=4))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Get class labels for plotting from the label encoder
class_labels = label_encoder.inverse_transform(np.arange(len(label_encoder.classes_)))

# Generate the confusion matrix
cm = confusion_matrix(y_test_original, y_pred_fine_tuned, labels=class_labels)

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
plt.title('Confusion Matrix for Baseline (Batch Size 16, Learning Rate 2e-5)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

### **Model Training and Evaluation with Oversampling**

In [ ]:
# Re-encode labels for consistency, though label_encoder is already defined
# This ensures the new blocks use the correct encoding if run independently
label_encoder_balanced = LabelEncoder()
y_train_encoded_balanced = label_encoder_balanced.fit_transform(y_train)
y_val_encoded_balanced = label_encoder_balanced.transform(y_val)
y_test_encoded_balanced = label_encoder_balanced.transform(y_test)

# Convert resampled data to PyTorch tensors
X_train_oversampled_tensor = torch.tensor(X_train_resampled_embeddings.values, dtype=torch.float32)
y_train_oversampled_tensor = torch.tensor(label_encoder_balanced.transform(y_train_resampled), dtype=torch.long)

# Create DataLoader for oversampled training data
train_dataset_oversampled = TensorDataset(X_train_oversampled_tensor, y_train_oversampled_tensor)
train_loader_oversampled = DataLoader(train_dataset_oversampled, batch_size=16, shuffle=True)

# Re-instantiate the model for oversampling (to ensure fresh weights)
model_oversampled = BertClassifier(input_dim, num_classes)
criterion_oversampled = nn.CrossEntropyLoss()
optimizer_oversampled = optim.Adam(model_oversampled.parameters(), lr=0.00002)

print("\n--- Training Model with Oversampled Data ---")

In [ ]:
num_epochs = 10
for epoch in range(num_epochs):
    model_oversampled.train()
    total_train_loss = 0
    for batch_X, batch_y in train_loader_oversampled:
        optimizer_oversampled.zero_grad()
        outputs = model_oversampled(batch_X)
        loss = criterion_oversampled(outputs, batch_y)
        loss.backward()
        optimizer_oversampled.step()
        total_train_loss += loss.item()

    model_oversampled.eval()
    total_val_loss = 0
    val_correct = 0
    total_val_samples = 0
    with torch.no_grad():
        for batch_X_val, batch_y_val in val_loader:
            outputs_val = model_oversampled(batch_X_val)
            loss_val = criterion_oversampled(outputs_val, batch_y_val)
            total_val_loss += loss_val.item()
            _, predicted_val = torch.max(outputs_val.data, 1)
            total_val_samples += batch_y_val.size(0)
            val_correct += (predicted_val == batch_y_val).sum().item()
    val_accuracy = val_correct / total_val_samples
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss (Oversampled): {total_train_loss/len(train_loader_oversampled):.4f}, Val Loss: {total_val_loss/len(val_loader):.4f}, Val Accuracy: {val_accuracy:.4f}")

print("Oversampling model training complete.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Data extracted from the output of the oversampling training cell (4bb7d1c4)
epochs_oversampled = range(1, 11)
train_losses_oversampled = [0.9652, 0.8800, 0.8406, 0.8095, 0.7875, 0.7643, 0.7529, 0.7371, 0.7244, 0.7127]
val_losses_oversampled = [0.9291, 0.8347, 0.8314, 0.8239, 0.7610, 0.8193, 0.7938, 0.7574, 0.7548, 0.7581]
val_accuracies_oversampled = [0.5630, 0.6415, 0.6443, 0.6485, 0.7017, 0.6597, 0.6709, 0.6891, 0.6919, 0.6807]

plt.figure(figsize=(12, 5))

# Plotting Loss
plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
plt.plot(epochs_oversampled, train_losses_oversampled, label='Training Loss (Oversampled)')
plt.plot(epochs_oversampled, val_losses_oversampled, label='Validation Loss')
plt.title('Training and Validation Loss per Epoch (Oversampling (BS 16, LR 2e-5))')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plotting Accuracy
plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
plt.plot(epochs_oversampled, val_accuracies_oversampled, label='Validation Accuracy', color='green')
plt.title('Validation Accuracy per Epoch (Oversampling (BS 16, LR 2e-5))')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# --- Evaluation with Oversampling Model ---
model_oversampled.eval()
all_preds_oversampled = []
all_labels_oversampled = []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        outputs = model_oversampled(batch_X)
        _, predicted = torch.max(outputs.data, 1)
        all_preds_oversampled.extend(predicted.cpu().numpy())
        all_labels_oversampled.extend(batch_y.cpu().numpy())

y_pred_oversampled = label_encoder_balanced.inverse_transform(all_preds_oversampled)
y_test_original_oversampled = label_encoder_balanced.inverse_transform(all_labels_oversampled)

print("\nEvaluation Results (Oversampling (BS 16, LR 2e-5)):")
print(f"Accuracy: {accuracy_score(y_test_original_oversampled, y_pred_oversampled):.4f}")
print(f"Precision: {precision_score(y_test_original_oversampled, y_pred_oversampled, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test_original_oversampled, y_pred_oversampled, average='weighted'):.4f}")
print(f"F1-Score: {f1_score(y_test_original_oversampled, y_pred_oversampled, average='weighted'):.4f}")
print("Classification Report:")
print(classification_report(y_test_original_oversampled, y_pred_oversampled, target_names=label_encoder_balanced.classes_, digits=4))

In [ ]:
# Confusion Matrix for Oversampling
class_labels_balanced = label_encoder_balanced.inverse_transform(np.arange(len(label_encoder_balanced.classes_)))
cm_oversampled = confusion_matrix(y_test_original_oversampled, y_pred_oversampled, labels=class_labels_balanced)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_oversampled, annot=True, fmt='d', cmap='Greens', xticklabels=class_labels_balanced, yticklabels=class_labels_balanced)
plt.title('Confusion Matrix (Oversampling (Batch Size 16, Learning Rate 2e-5))')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

### **Model Training and Evaluation with Undersampling**

In [ ]:
# Convert undersampled data to PyTorch tensors
X_train_undersampled_tensor = torch.tensor(X_train_undersampled_embeddings.values, dtype=torch.float32)
y_train_undersampled_tensor = torch.tensor(label_encoder_balanced.transform(y_train_undersampled), dtype=torch.long)

# Create DataLoader for undersampled training data
train_dataset_undersampled = TensorDataset(X_train_undersampled_tensor, y_train_undersampled_tensor)
train_loader_undersampled = DataLoader(train_dataset_undersampled, batch_size=16, shuffle=True)

# Re-instantiate the model for undersampling (to ensure fresh weights)
model_undersampled = BertClassifier(input_dim, num_classes)
criterion_undersampled = nn.CrossEntropyLoss()
optimizer_undersampled = optim.Adam(model_undersampled.parameters(), lr=0.00002)

print("\n--- Training Model with Undersampled Data ---")

In [ ]:
num_epochs = 10
for epoch in range(num_epochs):
    model_undersampled.train()
    total_train_loss = 0
    for batch_X, batch_y in train_loader_undersampled:
        optimizer_undersampled.zero_grad()
        outputs = model_undersampled(batch_X)
        loss = criterion_undersampled(outputs, batch_y)
        loss.backward()
        optimizer_undersampled.step()
        total_train_loss += loss.item()

    model_undersampled.eval()
    total_val_loss = 0
    val_correct = 0
    total_val_samples = 0
    with torch.no_grad():
        for batch_X_val, batch_y_val in val_loader:
            outputs_val = model_undersampled(batch_X_val)
            loss_val = criterion_undersampled(outputs_val, batch_y_val)
            total_val_loss += loss_val.item()
            _, predicted_val = torch.max(outputs_val.data, 1)
            total_val_samples += batch_y_val.size(0)
            val_correct += (predicted_val == batch_y_val).sum().item()
    val_accuracy = val_correct / total_val_samples
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss (Undersampled): {total_train_loss/len(train_loader_undersampled):.4f}, Val Loss: {total_val_loss/len(val_loader):.4f}, Val Accuracy: {val_accuracy:.4f}")

print("Undersampling model training complete.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Data extracted from the output of the undersampling training cell (f9d54d86)
epochs_undersampled = range(1, 11)
train_losses_undersampled = [1.0303, 0.9758, 0.9424, 0.9210, 0.8980, 0.8883, 0.8746, 0.8624, 0.8576, 0.8373]
val_losses_undersampled = [0.9888, 0.9643, 0.9054, 0.8874, 0.9244, 0.8707, 0.9129, 0.8692, 0.8552, 0.8299]
val_accuracies_undersampled = [0.5798, 0.5798, 0.6078, 0.6092, 0.5616, 0.6078, 0.5686, 0.6106, 0.6261, 0.6541]

plt.figure(figsize=(12, 5))

# Plotting Loss
plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
plt.plot(epochs_undersampled, train_losses_undersampled, label='Training Loss (Undersampled)')
plt.plot(epochs_undersampled, val_losses_undersampled, label='Validation Loss')
plt.title('Training and Validation Loss per Epoch (Undersampling (BS 16, LR 2e-5))')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plotting Accuracy
plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
plt.plot(epochs_undersampled, val_accuracies_undersampled, label='Validation Accuracy', color='green')
plt.title('Validation Accuracy per Epoch (Undersampling (BS 16, LR 2e-5))')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# --- Evaluation with Undersampling Model ---
model_undersampled.eval()
all_preds_undersampled = []
all_labels_undersampled = []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        outputs = model_undersampled(batch_X)
        _, predicted = torch.max(outputs.data, 1)
        all_preds_undersampled.extend(predicted.cpu().numpy())
        all_labels_undersampled.extend(batch_y.cpu().numpy())

y_pred_undersampled = label_encoder_balanced.inverse_transform(all_preds_undersampled)
y_test_original_undersampled = label_encoder_balanced.inverse_transform(all_labels_undersampled)

print("\nEvaluation Results (Undersampling (BS 16, LR 2e-5)):")
print(f"Accuracy: {accuracy_score(y_test_original_undersampled, y_pred_undersampled):.4f}")
print(f"Precision: {precision_score(y_test_original_undersampled, y_pred_undersampled, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test_original_undersampled, y_pred_undersampled, average='weighted'):.4f}")
print(f"F1-Score: {f1_score(y_test_original_undersampled, y_pred_undersampled, average='weighted'):.4f}")
print("Classification Report:")
print(classification_report(y_test_original_undersampled, y_pred_undersampled, target_names=label_encoder_balanced.classes_, digits=4))

In [ ]:
# Confusion Matrix for Undersampling
cm_undersampled = confusion_matrix(y_test_original_undersampled, y_pred_undersampled, labels=class_labels_balanced)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_undersampled, annot=True, fmt='d', cmap='Oranges', xticklabels=class_labels_balanced, yticklabels=class_labels_balanced)
plt.title('Confusion Matrix (Undersampling(Batch Size 16, Learning Rate 2e-5))')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

## **Batch Size 32 Learning Rate 2e-5**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

# 1. Encode labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train_embeddings.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_encoded, dtype=torch.long)
X_val_tensor = torch.tensor(X_val_embeddings.values, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_encoded, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_embeddings.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_encoded, dtype=torch.long)

In [ ]:
# 2. Define the Neural Network Classifier
class BertClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dim=256):
        super(BertClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim) # Dense layer 1
        self.relu = nn.ReLU()                       # Activation function
        self.dropout = nn.Dropout(0.3)              # Dropout for regularization
        self.fc2 = nn.Linear(hidden_dim, num_classes) # Dense layer 2 (output layer)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        # Softmax is typically applied by CrossEntropyLoss, so we return logits
        return x

# Instantiate the model
input_dim = X_train_embeddings.shape[1] # 768 for IndoBERT base embeddings
num_classes = len(label_encoder.classes_) # 3 (Negative, Neutral, Positive)
model = BertClassifier(input_dim, num_classes)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00002)

# Create DataLoader
batch_size = 32
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# 3. Training loop with Validation
num_epochs = 10
print("Starting fine-tuning with custom dense layers...")

for epoch in range(num_epochs):
    model.train() # Set model to training mode
    total_train_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad() # Clear gradients
        outputs = model(batch_X) # Forward pass
        loss = criterion(outputs, batch_y) # Compute loss
        loss.backward() # Backward pass
        optimizer.step() # Update weights
        total_train_loss += loss.item()

    # Validation step
    model.eval() # Set model to evaluation mode
    total_val_loss = 0
    val_correct = 0
    total_val_samples = 0
    with torch.no_grad(): # Disable gradient calculation during evaluation
        for batch_X_val, batch_y_val in val_loader:
            outputs_val = model(batch_X_val)
            loss_val = criterion(outputs_val, batch_y_val)
            total_val_loss += loss_val.item()
            _, predicted_val = torch.max(outputs_val.data, 1) # Get the class with the highest probability
            total_val_samples += batch_y_val.size(0)
            val_correct += (predicted_val == batch_y_val).sum().item()

    val_accuracy = val_correct / total_val_samples

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {total_train_loss/len(train_loader):.4f}, Val Loss: {total_val_loss/len(val_loader):.4f}, Val Accuracy: {val_accuracy:.4f}")

print("Fine-tuning complete.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Data extracted from the output of the training cell (mlldKnMTx4bt)
epochs_bs32_lr2e5 = range(1, 11)
train_losses_bs32_lr2e5 = [0.8556, 0.7903, 0.7647, 0.7432, 0.7255, 0.7094, 0.7092, 0.6956, 0.6882, 0.6742]
val_losses_bs32_lr2e5 = [0.8062, 0.7766, 0.7585, 0.7445, 0.7324, 0.7228, 0.7144, 0.7095, 0.7043, 0.6977]
val_accuracies_bs32_lr2e5 = [0.6779, 0.6891, 0.6863, 0.7003, 0.7017, 0.7157, 0.7227, 0.7227, 0.7241, 0.7297]

plt.figure(figsize=(12, 5))

# Plotting Loss
plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
plt.plot(epochs_bs32_lr2e5, train_losses_bs32_lr2e5, label='Training Loss')
plt.plot(epochs_bs32_lr2e5, val_losses_bs32_lr2e5, label='Validation Loss')
plt.title('Training and Validation Loss per Epoch (BS 32, LR 2e-5)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plotting Accuracy
plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
plt.plot(epochs_bs32_lr2e5, val_accuracies_bs32_lr2e5, label='Validation Accuracy', color='green')
plt.title('Validation Accuracy per Epoch (BS 32, LR 2e-5)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# 4. Evaluation
model.eval() # Set model to evaluation mode
all_preds = []
all_labels = []

with torch.no_grad(): # Disable gradient calculation during evaluation
    for batch_X, batch_y in test_loader:
        outputs = model(batch_X)
        _, predicted = torch.max(outputs.data, 1) # Get the class with the highest probability
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())

# Convert numerical predictions back to original labels for classification report
y_pred_fine_tuned = label_encoder.inverse_transform(all_preds)
y_test_original = label_encoder.inverse_transform(all_labels)

print("\nEvaluation Results (BS 32, LR 2e-5)):")
print(f"Accuracy: {accuracy_score(y_test_original, y_pred_fine_tuned):.4f}")

# Calculate and print additional metrics explicitly
from sklearn.metrics import precision_score, recall_score, f1_score

print(f"Precision: {precision_score(y_test_original, y_pred_fine_tuned, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test_original, y_pred_fine_tuned, average='weighted'):.4f}")
print(f"F1-Score: {f1_score(y_test_original, y_pred_fine_tuned, average='weighted'):.4f}")

print("\nClassification Report:")
print(classification_report(y_test_original, y_pred_fine_tuned, target_names=label_encoder.classes_, digits=4))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Get class labels for plotting from the label encoder
class_labels = label_encoder.inverse_transform(np.arange(len(label_encoder.classes_)))

# Generate the confusion matrix
cm = confusion_matrix(y_test_original, y_pred_fine_tuned, labels=class_labels)

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
plt.title('Confusion Matrix (Batch Size 32, Learning Rate 2e-5)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

### **Model Training and Evaluation with Oversampling**

In [ ]:
# Re-encode labels for consistency, though label_encoder is already defined
# This ensures the new blocks use the correct encoding if run independently
label_encoder_balanced = LabelEncoder()
y_train_encoded_balanced = label_encoder_balanced.fit_transform(y_train)
y_val_encoded_balanced = label_encoder_balanced.transform(y_val)
y_test_encoded_balanced = label_encoder_balanced.transform(y_test)

# Convert resampled data to PyTorch tensors
X_train_oversampled_tensor = torch.tensor(X_train_resampled_embeddings.values, dtype=torch.float32)
y_train_oversampled_tensor = torch.tensor(label_encoder_balanced.transform(y_train_resampled), dtype=torch.long)

# Create DataLoader for oversampled training data
train_dataset_oversampled = TensorDataset(X_train_oversampled_tensor, y_train_oversampled_tensor)
train_loader_oversampled = DataLoader(train_dataset_oversampled, batch_size=32, shuffle=True)

# Re-instantiate the model for oversampling (to ensure fresh weights)
model_oversampled = BertClassifier(input_dim, num_classes)
criterion_oversampled = nn.CrossEntropyLoss()
optimizer_oversampled = optim.Adam(model_oversampled.parameters(), lr=0.00002)

print("\n--- Training Model with Oversampled Data ---")

In [ ]:
num_epochs = 10
for epoch in range(num_epochs):
    model_oversampled.train()
    total_train_loss = 0
    for batch_X, batch_y in train_loader_oversampled:
        optimizer_oversampled.zero_grad()
        outputs = model_oversampled(batch_X)
        loss = criterion_oversampled(outputs, batch_y)
        loss.backward()
        optimizer_oversampled.step()
        total_train_loss += loss.item()

    model_oversampled.eval()
    total_val_loss = 0
    val_correct = 0
    total_val_samples = 0
    with torch.no_grad():
        for batch_X_val, batch_y_val in val_loader:
            outputs_val = model_oversampled(batch_X_val)
            loss_val = criterion_oversampled(outputs_val, batch_y_val)
            total_val_loss += loss_val.item()
            _, predicted_val = torch.max(outputs_val.data, 1)
            total_val_samples += batch_y_val.size(0)
            val_correct += (predicted_val == batch_y_val).sum().item()
    val_accuracy = val_correct / total_val_samples
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss (Oversampled): {total_train_loss/len(train_loader_oversampled):.4f}, Val Loss: {total_val_loss/len(val_loader):.4f}, Val Accuracy: {val_accuracy:.4f}")

print("Oversampling model training complete.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Data extracted from the output of the oversampling training cell (i8sYVRIbyLD7)
epochs_oversampled_bs32_lr2e5 = range(1, 11)
train_losses_oversampled_bs32_lr2e5 = [0.9907, 0.9110, 0.8708, 0.8439, 0.8199, 0.8012, 0.7865, 0.7689, 0.7569, 0.7454]
val_losses_oversampled_bs32_lr2e5 = [0.9266, 0.9015, 0.8702, 0.8185, 0.8214, 0.8675, 0.7888, 0.7907, 0.8172, 0.7750]
val_accuracies_oversampled_bs32_lr2e5 = [0.5952, 0.6064, 0.6261, 0.6667, 0.6709, 0.6246, 0.6891, 0.6751, 0.6527, 0.6919]

plt.figure(figsize=(12, 5))

# Plotting Loss
plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
plt.plot(epochs_oversampled_bs32_lr2e5, train_losses_oversampled_bs32_lr2e5, label='Training Loss (Oversampled)')
plt.plot(epochs_oversampled_bs32_lr2e5, val_losses_oversampled_bs32_lr2e5, label='Validation Loss')
plt.title('Training and Validation Loss per Epoch (Oversampling BS 32, LR 2e-5)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plotting Accuracy
plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
plt.plot(epochs_oversampled_bs32_lr2e5, val_accuracies_oversampled_bs32_lr2e5, label='Validation Accuracy', color='green')
plt.title('Validation Accuracy per Epoch (Oversampling BS 32, LR 2e-5)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# --- Evaluation with Oversampling Model ---
model_oversampled.eval()
all_preds_oversampled = []
all_labels_oversampled = []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        outputs = model_oversampled(batch_X)
        _, predicted = torch.max(outputs.data, 1)
        all_preds_oversampled.extend(predicted.cpu().numpy())
        all_labels_oversampled.extend(batch_y.cpu().numpy())

y_pred_oversampled = label_encoder_balanced.inverse_transform(all_preds_oversampled)
y_test_original_oversampled = label_encoder_balanced.inverse_transform(all_labels_oversampled)

print("\nEvaluation Results Oversampling (BS 32, LR 2e-5):")
print(f"Accuracy: {accuracy_score(y_test_original_oversampled, y_pred_oversampled):.4f}")
print(f"Precision: {precision_score(y_test_original_oversampled, y_pred_oversampled, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test_original_oversampled, y_pred_oversampled, average='weighted'):.4f}")
print(f"F1-Score: {f1_score(y_test_original_oversampled, y_pred_oversampled, average='weighted'):.4f}")
print("Classification Report:")
print(classification_report(y_test_original_oversampled, y_pred_oversampled, target_names=label_encoder_balanced.classes_, digits=4))

In [ ]:
# Confusion Matrix for Oversampling
class_labels_balanced = label_encoder_balanced.inverse_transform(np.arange(len(label_encoder_balanced.classes_)))
cm_oversampled = confusion_matrix(y_test_original_oversampled, y_pred_oversampled, labels=class_labels_balanced)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_oversampled, annot=True, fmt='d', cmap='Greens', xticklabels=class_labels_balanced, yticklabels=class_labels_balanced)
plt.title('Confusion Matrix (Oversampling (Batch Size 32, Learning Rate 2e-5))')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

### **Model Training and Evaluation with Undersampling**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Convert undersampled data to PyTorch tensors
X_train_undersampled_tensor = torch.tensor(X_train_undersampled_embeddings.values, dtype=torch.float32)
y_train_undersampled_tensor = torch.tensor(label_encoder_balanced.transform(y_train_undersampled), dtype=torch.long)

# Create DataLoader for undersampled training data
train_dataset_undersampled = TensorDataset(X_train_undersampled_tensor, y_train_undersampled_tensor)
train_loader_undersampled = DataLoader(train_dataset_undersampled, batch_size=32, shuffle=True)

# Re-instantiate the model for undersampling (to ensure fresh weights)
# Assuming BertClassifier, input_dim, and num_classes are defined in previous cells
model_undersampled = BertClassifier(input_dim, num_classes)
criterion_undersampled = nn.CrossEntropyLoss()
optimizer_undersampled = optim.Adam(model_undersampled.parameters(), lr=0.00002)

print("\n--- Training Model with Undersampled Data ---")
num_epochs = 10
for epoch in range(num_epochs):
    model_undersampled.train()
    total_train_loss = 0
    for batch_X, batch_y in train_loader_undersampled:
        optimizer_undersampled.zero_grad()
        outputs = model_undersampled(batch_X)
        loss = criterion_undersampled(outputs, batch_y)
        loss.backward()
        optimizer_undersampled.step()
        total_train_loss += loss.item()

    model_undersampled.eval()
    total_val_loss = 0
    val_correct = 0
    total_val_samples = 0
    with torch.no_grad():
        for batch_X_val, batch_y_val in val_loader:
            outputs_val = model_undersampled(batch_X_val)
            loss_val = criterion_undersampled(outputs_val, batch_y_val)
            total_val_loss += loss_val.item()
            _, predicted_val = torch.max(outputs_val.data, 1)
            total_val_samples += batch_y_val.size(0)
            val_correct += (predicted_val == batch_y_val).sum().item()
    val_accuracy = val_correct / total_val_samples
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss (Undersampled): {total_train_loss/len(train_loader_undersampled):.4f}, Val Loss: {total_val_loss/len(val_loader):.4f}, Val Accuracy: {val_accuracy:.4f}")

print("Undersampling model training complete.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Data extracted from the output of the undersampling training cell (P2HGrT0Pyjcc)
epochs_undersampled_bs32_lr2e5 = range(1, 11)
train_losses_undersampled_bs32_lr2e5 = [1.0444, 0.9794, 0.9481, 0.9330, 0.9180, 0.9066, 0.8908, 0.8831, 0.8777, 0.8723]
val_losses_undersampled_bs32_lr2e5 = [1.0303, 0.9875, 0.9317, 0.9458, 0.9253, 0.9217, 0.9295, 0.8941, 0.8881, 0.8678]
val_accuracies_undersampled_bs32_lr2e5 = [0.5616, 0.5756, 0.5966, 0.5490, 0.5602, 0.5560, 0.5728, 0.5910, 0.5994, 0.5966]

plt.figure(figsize=(12, 5))

# Plotting Loss
plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
plt.plot(epochs_undersampled_bs32_lr2e5, train_losses_undersampled_bs32_lr2e5, label='Training Loss (Undersampled)')
plt.plot(epochs_undersampled_bs32_lr2e5, val_losses_undersampled_bs32_lr2e5, label='Validation Loss')
plt.title('Training and Validation Loss per Epoch (Undersampling BS 32, LR 2e-5)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plotting Accuracy
plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
plt.plot(epochs_undersampled_bs32_lr2e5, val_accuracies_undersampled_bs32_lr2e5, label='Validation Accuracy', color='green')
plt.title('Validation Accuracy per Epoch (Undersampling BS 32, LR 2e-5)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# --- Evaluation with Undersampling Model ---
model_undersampled.eval()
all_preds_undersampled = []
all_labels_undersampled = []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        outputs = model_undersampled(batch_X)
        _, predicted = torch.max(outputs.data, 1)
        all_preds_undersampled.extend(predicted.cpu().numpy())
        all_labels_undersampled.extend(batch_y.cpu().numpy())

y_pred_undersampled = label_encoder_balanced.inverse_transform(all_preds_undersampled)
y_test_original_undersampled = label_encoder_balanced.inverse_transform(all_labels_undersampled)

print("\nEvaluation Results (Undersampling (BS 32, LR 2e-5)):")
print(f"Accuracy: {accuracy_score(y_test_original_undersampled, y_pred_undersampled):.4f}")
print(f"Precision: {precision_score(y_test_original_undersampled, y_pred_undersampled, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test_original_undersampled, y_pred_undersampled, average='weighted'):.4f}")
print(f"F1-Score: {f1_score(y_test_original_undersampled, y_pred_undersampled, average='weighted'):.4f}")
print("Classification Report:")
print(classification_report(y_test_original_undersampled, y_pred_undersampled, target_names=label_encoder_balanced.classes_, digits=4))

# Confusion Matrix for Undersampling
class_labels_balanced = label_encoder_balanced.inverse_transform(np.arange(len(label_encoder_balanced.classes_)))
cm_undersampled = confusion_matrix(y_test_original_undersampled, y_pred_undersampled, labels=class_labels_balanced)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_undersampled, annot=True, fmt='d', cmap='Oranges', xticklabels=class_labels_balanced, yticklabels=class_labels_balanced)
plt.title('Confusion Matrix (Oversampling (Batch Size 32, Learning Rate 2e-5))')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

## **Batch Size 16 Learning Rate 1e-5**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Encode labels
# Reuse label_encoder as it's already fitted to y_train
label_encoder_bs16_lr1e5 = LabelEncoder()
y_train_encoded_bs16_lr1e5 = label_encoder_bs16_lr1e5.fit_transform(y_train)
y_val_encoded_bs16_lr1e5 = label_encoder_bs16_lr1e5.transform(y_val)
y_test_encoded_bs16_lr1e5 = label_encoder_bs16_lr1e5.transform(y_test)

# Convert to PyTorch tensors
X_train_tensor_bs16_lr1e5 = torch.tensor(X_train_embeddings.values, dtype=torch.float32)
y_train_tensor_bs16_lr1e5 = torch.tensor(y_train_encoded_bs16_lr1e5, dtype=torch.long)
X_val_tensor_bs16_lr1e5 = torch.tensor(X_val_embeddings.values, dtype=torch.float32)
y_val_tensor_bs16_lr1e5 = torch.tensor(y_val_encoded_bs16_lr1e5, dtype=torch.long)
X_test_tensor_bs16_lr1e5 = torch.tensor(X_test_embeddings.values, dtype=torch.float32)
y_test_tensor_bs16_lr1e5 = torch.tensor(y_test_encoded_bs16_lr1e5, dtype=torch.long)

In [ ]:
# 2. Define the Neural Network Classifier (re-defined for self-containment)
class BertClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dim=256):
        super(BertClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Instantiate the model with new weights
model_bs16_lr1e5 = BertClassifier(input_dim, num_classes)

# Define loss function and optimizer with LR = 1e-5
criterion_bs16_lr1e5 = nn.CrossEntropyLoss()
optimizer_bs16_lr1e5 = optim.Adam(model_bs16_lr1e5.parameters(), lr=0.00001) # Learning rate changed to 1e-5

# Create DataLoaders with batch_size = 16
batch_size_bs16_lr1e5 = 16 # Batch size changed to 16
train_dataset_bs16_lr1e5 = TensorDataset(X_train_tensor_bs16_lr1e5, y_train_tensor_bs16_lr1e5)
train_loader_bs16_lr1e5 = DataLoader(train_dataset_bs16_lr1e5, batch_size=batch_size_bs16_lr1e5, shuffle=True)

val_dataset_bs16_lr1e5 = TensorDataset(X_val_tensor_bs16_lr1e5, y_val_tensor_bs16_lr1e5)
val_loader_bs16_lr1e5 = DataLoader(val_dataset_bs16_lr1e5, batch_size=batch_size_bs16_lr1e5, shuffle=False)

test_dataset_bs16_lr1e5 = TensorDataset(X_test_tensor_bs16_lr1e5, y_test_tensor_bs16_lr1e5)
test_loader_bs16_lr1e5 = DataLoader(test_dataset_bs16_lr1e5, batch_size=batch_size_bs16_lr1e5, shuffle=False)

In [ ]:
# 3. Training loop with Validation
num_epochs = 10
print("Starting fine-tuning with Batch Size 16, Learning Rate 1e-5...")

for epoch in range(num_epochs):
    model_bs16_lr1e5.train() # Set model to training mode
    total_train_loss = 0
    for batch_X, batch_y in train_loader_bs16_lr1e5:
        optimizer_bs16_lr1e5.zero_grad() # Clear gradients
        outputs = model_bs16_lr1e5(batch_X) # Forward pass
        loss = criterion_bs16_lr1e5(outputs, batch_y) # Compute loss
        loss.backward() # Backward pass
        optimizer_bs16_lr1e5.step() # Update weights
        total_train_loss += loss.item()

    # Validation step
    model_bs16_lr1e5.eval() # Set model to evaluation mode
    total_val_loss = 0
    val_correct = 0
    total_val_samples = 0
    with torch.no_grad(): # Disable gradient calculation during evaluation
        for batch_X_val, batch_y_val in val_loader_bs16_lr1e5:
            outputs_val = model_bs16_lr1e5(batch_X_val)
            loss_val = criterion_bs16_lr1e5(outputs_val, batch_y_val)
            total_val_loss += loss_val.item()
            _, predicted_val = torch.max(outputs_val.data, 1) # Get the class with the highest probability
            total_val_samples += batch_y_val.size(0)
            val_correct += (predicted_val == batch_y_val).sum().item()

    val_accuracy = val_correct / total_val_samples

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {total_train_loss/len(train_loader_bs16_lr1e5):.4f}, Val Loss: {total_val_loss/len(val_loader_bs16_lr1e5):.4f}, Val Accuracy: {val_accuracy:.4f}")

print("Fine-tuning complete for Batch Size 16, Learning Rate 1e-5.")

In [ ]:
import matplotlib.pyplot as plt

# Data from training cell Zucg9vP-ywxk (BS 16, LR 1e-5)
epochs = range(1, 11)
train_losses_bs16_lr1e5 = [0.9184, 0.8016, 0.7766, 0.7659, 0.7435, 0.7350, 0.7275, 0.7235, 0.7106, 0.6967]
val_losses_bs16_lr1e5 = [0.8268, 0.7895, 0.7734, 0.7611, 0.7501, 0.7406, 0.7320, 0.7248, 0.7182, 0.7119]
val_accuracies_bs16_lr1e5 = [0.6443, 0.6793, 0.6821, 0.6849, 0.6919, 0.7017, 0.7045, 0.7115, 0.7157, 0.7157]

plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(epochs, train_losses_bs16_lr1e5, label='Training Loss')
plt.plot(epochs, val_losses_bs16_lr1e5, label='Validation Loss')
plt.title('Loss: (BS 16, LR 1e-5)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, val_accuracies_bs16_lr1e5, label='Validation Accuracy', color='green')
plt.title('Accuracy: (BS 16, LR 1e-5)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# 4. Evaluation
model_bs16_lr1e5.eval() # Set model to evaluation mode
all_preds_bs16_lr1e5 = []
all_labels_bs16_lr1e5 = []

with torch.no_grad(): # Disable gradient calculation during evaluation
    for batch_X, batch_y in test_loader_bs16_lr1e5:
        outputs = model_bs16_lr1e5(batch_X)
        _, predicted = torch.max(outputs.data, 1) # Get the class with the highest probability
        all_preds_bs16_lr1e5.extend(predicted.cpu().numpy())
        all_labels_bs16_lr1e5.extend(batch_y.cpu().numpy())

# Convert numerical predictions back to original labels for classification report
y_pred_bs16_lr1e5 = label_encoder_bs16_lr1e5.inverse_transform(all_preds_bs16_lr1e5)
y_test_original_bs16_lr1e5 = label_encoder_bs16_lr1e5.inverse_transform(all_labels_bs16_lr1e5)

print("\nEvaluation Results (Batch Size 16, Learning Rate 1e-5):")
print(f"Accuracy: {accuracy_score(y_test_original_bs16_lr1e5, y_pred_bs16_lr1e5):.4f}")
print(f"Precision: {precision_score(y_test_original_bs16_lr1e5, y_pred_bs16_lr1e5, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test_original_bs16_lr1e5, y_pred_bs16_lr1e5, average='weighted'):.4f}")
print(f"F1-Score: {f1_score(y_test_original_bs16_lr1e5, y_pred_bs16_lr1e5, average='weighted'):.4f}")

print("\nClassification Report:")
print(classification_report(y_test_original_bs16_lr1e5, y_pred_bs16_lr1e5, target_names=label_encoder_bs16_lr1e5.classes_, digits=4))

In [ ]:
# Confusion Matrix for Batch Size 16, Learning Rate 1e-5
class_labels_bs16_lr1e5 = label_encoder_bs16_lr1e5.inverse_transform(np.arange(len(label_encoder_bs16_lr1e5.classes_)))
cm_bs16_lr1e5 = confusion_matrix(y_test_original_bs16_lr1e5, y_pred_bs16_lr1e5, labels=class_labels_bs16_lr1e5)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_bs16_lr1e5, annot=True, fmt='d', cmap='Purples', xticklabels=class_labels_bs16_lr1e5, yticklabels=class_labels_bs16_lr1e5)
plt.title('Confusion Matrix for Batch Size 16, Learning Rate 1e-5')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

### **Batch Size 16 Learning Rate 1e-5 with Oversampling**

In [ ]:
# Re-encode labels for consistency, though label_encoder_balanced is already defined
# This ensures the new blocks use the correct encoding if run independently
label_encoder_balanced_bs16_lr1e5_oversampled = LabelEncoder()
y_train_encoded_balanced_bs16_lr1e5_oversampled = label_encoder_balanced_bs16_lr1e5_oversampled.fit_transform(y_train)
y_val_encoded_balanced_bs16_lr1e5_oversampled = label_encoder_balanced_bs16_lr1e5_oversampled.transform(y_val)
y_test_encoded_balanced_bs16_lr1e5_oversampled = label_encoder_balanced_bs16_lr1e5_oversampled.transform(y_test)

# Convert resampled data to PyTorch tensors
X_train_oversampled_tensor_bs16_lr1e5 = torch.tensor(X_train_resampled_embeddings.values, dtype=torch.float32)
y_train_oversampled_tensor_bs16_lr1e5 = torch.tensor(label_encoder_balanced_bs16_lr1e5_oversampled.transform(y_train_resampled), dtype=torch.long)

# Create DataLoader for oversampled training data
train_dataset_oversampled_bs16_lr1e5 = TensorDataset(X_train_oversampled_tensor_bs16_lr1e5, y_train_oversampled_tensor_bs16_lr1e5)
train_loader_oversampled_bs16_lr1e5 = DataLoader(train_dataset_oversampled_bs16_lr1e5, batch_size=16, shuffle=True)

# Re-instantiate the model for oversampling (to ensure fresh weights)
model_oversampled_bs16_lr1e5 = BertClassifier(input_dim, num_classes)
criterion_oversampled_bs16_lr1e5 = nn.CrossEntropyLoss()
optimizer_oversampled_bs16_lr1e5 = optim.Adam(model_oversampled_bs16_lr1e5.parameters(), lr=0.00001)

print("\n--- Training Model with Oversampled Data (Batch Size 16, Learning Rate 1e-5) ---")
num_epochs = 10
for epoch in range(num_epochs):
    model_oversampled_bs16_lr1e5.train()
    total_train_loss = 0
    for batch_X, batch_y in train_loader_oversampled_bs16_lr1e5:
        optimizer_oversampled_bs16_lr1e5.zero_grad()
        outputs = model_oversampled_bs16_lr1e5(batch_X)
        loss = criterion_oversampled_bs16_lr1e5(outputs, batch_y)
        loss.backward()
        optimizer_oversampled_bs16_lr1e5.step()
        total_train_loss += loss.item()

    model_oversampled_bs16_lr1e5.eval()
    total_val_loss = 0
    val_correct = 0
    total_val_samples = 0
    with torch.no_grad():
        for batch_X_val, batch_y_val in val_loader:
            outputs_val = model_oversampled_bs16_lr1e5(batch_X_val)
            loss_val = criterion_oversampled_bs16_lr1e5(outputs_val, batch_y_val)
            total_val_loss += loss_val.item()
            _, predicted_val = torch.max(outputs_val.data, 1)
            total_val_samples += batch_y_val.size(0)
            val_correct += (predicted_val == batch_y_val).sum().item()
    val_accuracy = val_correct / total_val_samples
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss (Oversampled): {total_train_loss/len(train_loader_oversampled_bs16_lr1e5):.4f}, Val Loss: {total_val_loss/len(val_loader):.4f}, Val Accuracy: {val_accuracy:.4f}")

print("Oversampling model training complete (Batch Size 16, Learning Rate 1e-5).")

In [ ]:
import matplotlib.pyplot as plt

# Data from training cell xsGMfRD7y68D (Oversampling BS 16, LR 1e-5)
epochs = range(1, 11)
train_losses_over_bs16 = [0.9917, 0.9168, 0.8879, 0.8624, 0.8445, 0.8250, 0.8138, 0.7982, 0.7884, 0.7749]
val_losses_over_bs16 = [0.9588, 0.9261, 0.8960, 0.8962, 0.8595, 0.8568, 0.8146, 0.7993, 0.8104, 0.7916]
val_accuracies_over_bs16 = [0.5924, 0.5938, 0.6078, 0.5966, 0.6190, 0.6261, 0.6569, 0.6681, 0.6611, 0.6793]

plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(epochs, train_losses_over_bs16, label='Training Loss')
plt.plot(epochs, val_losses_over_bs16, label='Validation Loss')
plt.title('Loss: Oversampling (BS 16, LR 1e-5)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, val_accuracies_over_bs16, label='Validation Accuracy', color='green')
plt.title('Accuracy: Oversampling (BS 16, LR 1e-5)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# --- Evaluation with Oversampling Model (Batch Size 16, Learning Rate 1e-5) ---
model_oversampled_bs16_lr1e5.eval()
all_preds_oversampled_bs16_lr1e5 = []
all_labels_oversampled_bs16_lr1e5 = []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        outputs = model_oversampled_bs16_lr1e5(batch_X)
        _, predicted = torch.max(outputs.data, 1)
        all_preds_oversampled_bs16_lr1e5.extend(predicted.cpu().numpy())
        all_labels_oversampled_bs16_lr1e5.extend(batch_y.cpu().numpy())

y_pred_oversampled_bs16_lr1e5 = label_encoder_balanced_bs16_lr1e5_oversampled.inverse_transform(all_preds_oversampled_bs16_lr1e5)
y_test_original_oversampled_bs16_lr1e5 = label_encoder_balanced_bs16_lr1e5_oversampled.inverse_transform(all_labels_oversampled_bs16_lr1e5)

print("\nEvaluation Results (Oversampling (BS 16, LR 1e-5)):")
print(f"Accuracy: {accuracy_score(y_test_original_oversampled_bs16_lr1e5, y_pred_oversampled_bs16_lr1e5):.4f}")
print(f"Precision: {precision_score(y_test_original_oversampled_bs16_lr1e5, y_pred_oversampled_bs16_lr1e5, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test_original_oversampled_bs16_lr1e5, y_pred_oversampled_bs16_lr1e5, average='weighted'):.4f}")
print(f"F1-Score: {f1_score(y_test_original_oversampled_bs16_lr1e5, y_pred_oversampled_bs16_lr1e5, average='weighted'):.4f}")
print("Classification Report:")
print(classification_report(y_test_original_oversampled_bs16_lr1e5, y_pred_oversampled_bs16_lr1e5, target_names=label_encoder_balanced_bs16_lr1e5_oversampled.classes_, digits=4))

# Confusion Matrix for Oversampling
class_labels_balanced_bs16_lr1e5_oversampled = label_encoder_balanced_bs16_lr1e5_oversampled.inverse_transform(np.arange(len(label_encoder_balanced_bs16_lr1e5_oversampled.classes_)))
cm_oversampled_bs16_lr1e5 = confusion_matrix(y_test_original_oversampled_bs16_lr1e5, y_pred_oversampled_bs16_lr1e5, labels=class_labels_balanced_bs16_lr1e5_oversampled)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_oversampled_bs16_lr1e5, annot=True, fmt='d', cmap='Greens', xticklabels=class_labels_balanced_bs16_lr1e5_oversampled, yticklabels=class_labels_balanced_bs16_lr1e5_oversampled)
plt.title('Confusion Matrix (Oversampling (Batch Size 16, Learning Rate 1e-5))')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

### **Batch Size 16 Learning Rate 1e-5 with Undersampling**

In [ ]:
# Convert undersampled data to PyTorch tensors
X_train_undersampled_tensor_bs16_lr1e5 = torch.tensor(X_train_undersampled_embeddings.values, dtype=torch.float32)
y_train_undersampled_tensor_bs16_lr1e5 = torch.tensor(label_encoder_balanced_bs16_lr1e5_oversampled.transform(y_train_undersampled), dtype=torch.long)

# Create DataLoader for undersampled training data
train_dataset_undersampled_bs16_lr1e5 = TensorDataset(X_train_undersampled_tensor_bs16_lr1e5, y_train_undersampled_tensor_bs16_lr1e5)
train_loader_undersampled_bs16_lr1e5 = DataLoader(train_dataset_undersampled_bs16_lr1e5, batch_size=16, shuffle=True)

# Re-instantiate the model for undersampling (to ensure fresh weights)
model_undersampled_bs16_lr1e5 = BertClassifier(input_dim, num_classes)
criterion_undersampled_bs16_lr1e5 = nn.CrossEntropyLoss()
optimizer_undersampled_bs16_lr1e5 = optim.Adam(model_undersampled_bs16_lr1e5.parameters(), lr=0.00001)

print("\n--- Training Model with Undersampled Data (Batch Size 16, Learning Rate 1e-5) ---")
num_epochs = 10
for epoch in range(num_epochs):
    model_undersampled_bs16_lr1e5.train()
    total_train_loss = 0
    for batch_X, batch_y in train_loader_undersampled_bs16_lr1e5:
        optimizer_undersampled_bs16_lr1e5.zero_grad()
        outputs = model_undersampled_bs16_lr1e5(batch_X)
        loss = criterion_undersampled_bs16_lr1e5(outputs, batch_y)
        loss.backward()
        optimizer_undersampled_bs16_lr1e5.step()
        total_train_loss += loss.item()

    model_undersampled_bs16_lr1e5.eval()
    total_val_loss = 0
    val_correct = 0
    total_val_samples = 0
    with torch.no_grad():
        for batch_X_val, batch_y_val in val_loader:
            outputs_val = model_undersampled_bs16_lr1e5(batch_X_val)
            loss_val = criterion_undersampled_bs16_lr1e5(outputs_val, batch_y_val)
            total_val_loss += loss_val.item()
            _, predicted_val = torch.max(outputs_val.data, 1)
            total_val_samples += batch_y_val.size(0)
            val_correct += (predicted_val == batch_y_val).sum().item()
    val_accuracy = val_correct / total_val_samples
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss (Undersampled): {total_train_loss/len(train_loader_undersampled_bs16_lr1e5):.4f}, Val Loss: {total_val_loss/len(val_loader):.4f}, Val Accuracy: {val_accuracy:.4f}")

print("Undersampling model training complete (Batch Size 16, Learning Rate 1e-5).")

In [ ]:
import matplotlib.pyplot as plt

# Data from training cell FulAqIwVzJxE (Undersampling BS 16, LR 1e-5)
epochs = range(1, 11)
train_losses_under_bs16 = [1.0540, 0.9980, 0.9618, 0.9484, 0.9415, 0.9240, 0.9169, 0.9106, 0.8994, 0.8916]
val_losses_under_bs16 = [1.0068, 0.9632, 0.9567, 0.9605, 0.9427, 0.9294, 0.9322, 0.9130, 0.9158, 0.9230]
val_accuracies_under_bs16 = [0.5504, 0.5812, 0.5616, 0.5462, 0.5644, 0.5742, 0.5588, 0.5784, 0.5756, 0.5672]

plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(epochs, train_losses_under_bs16, label='Training Loss')
plt.plot(epochs, val_losses_under_bs16, label='Validation Loss')
plt.title('Loss: Undersampling (BS 16, LR 1e-5)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, val_accuracies_under_bs16, label='Validation Accuracy', color='green')
plt.title('Accuracy: Undersampling (BS 16, LR 1e-5)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# --- Evaluation with Undersampling Model (Batch Size 16, Learning Rate 1e-5) ---
model_undersampled_bs16_lr1e5.eval()
all_preds_undersampled_bs16_lr1e5 = []
all_labels_undersampled_bs16_lr1e5 = []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        outputs = model_undersampled_bs16_lr1e5(batch_X)
        _, predicted = torch.max(outputs.data, 1)
        all_preds_undersampled_bs16_lr1e5.extend(predicted.cpu().numpy())
        all_labels_undersampled_bs16_lr1e5.extend(batch_y.cpu().numpy())

y_pred_undersampled_bs16_lr1e5 = label_encoder_balanced_bs16_lr1e5_oversampled.inverse_transform(all_preds_undersampled_bs16_lr1e5)
y_test_original_undersampled_bs16_lr1e5 = label_encoder_balanced_bs16_lr1e5_oversampled.inverse_transform(all_labels_undersampled_bs16_lr1e5)

print("\nEvaluation Results (Undersampling (BS 16, LR 1e-5)):")
print(f"Accuracy: {accuracy_score(y_test_original_undersampled_bs16_lr1e5, y_pred_undersampled_bs16_lr1e5):.4f}")
print(f"Precision: {precision_score(y_test_original_undersampled_bs16_lr1e5, y_pred_undersampled_bs16_lr1e5, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test_original_undersampled_bs16_lr1e5, y_pred_undersampled_bs16_lr1e5, average='weighted'):.4f}")
print(f"F1-Score: {f1_score(y_test_original_undersampled_bs16_lr1e5, y_pred_undersampled_bs16_lr1e5, average='weighted'):.4f}")
print("Classification Report:")
print(classification_report(y_test_original_undersampled_bs16_lr1e5, y_pred_undersampled_bs16_lr1e5, target_names=label_encoder_balanced_bs16_lr1e5_oversampled.classes_, digits=4))

# Confusion Matrix for Undersampling
cm_undersampled_bs16_lr1e5 = confusion_matrix(y_test_original_undersampled_bs16_lr1e5, y_pred_undersampled_bs16_lr1e5, labels=class_labels_balanced_bs16_lr1e5_oversampled)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_undersampled_bs16_lr1e5, annot=True, fmt='d', cmap='Oranges', xticklabels=class_labels_balanced_bs16_lr1e5_oversampled, yticklabels=class_labels_balanced_bs16_lr1e5_oversampled)
plt.title('Confusion Matrix (Undersampling (Batch Size 16, Learning Rate 1e-5))')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

## **Batch Size 32 Learning Rate 1e-5**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Encode labels
# Reuse label_encoder as it's already fitted to y_train
label_encoder_bs32_lr1e5 = LabelEncoder()
y_train_encoded_bs32_lr1e5 = label_encoder_bs32_lr1e5.fit_transform(y_train)
y_val_encoded_bs32_lr1e5 = label_encoder_bs32_lr1e5.transform(y_val)
y_test_encoded_bs32_lr1e5 = label_encoder_bs32_lr1e5.transform(y_test)

# Convert to PyTorch tensors
X_train_tensor_bs32_lr1e5 = torch.tensor(X_train_embeddings.values, dtype=torch.float32)
y_train_tensor_bs32_lr1e5 = torch.tensor(y_train_encoded_bs32_lr1e5, dtype=torch.long)
X_val_tensor_bs32_lr1e5 = torch.tensor(X_val_embeddings.values, dtype=torch.float32)
y_val_tensor_bs32_lr1e5 = torch.tensor(y_val_encoded_bs32_lr1e5, dtype=torch.long)
X_test_tensor_bs32_lr1e5 = torch.tensor(X_test_embeddings.values, dtype=torch.float32)
y_test_tensor_bs32_lr1e5 = torch.tensor(y_test_encoded_bs32_lr1e5, dtype=torch.long)

In [ ]:
# 2. Define the Neural Network Classifier (re-defined for self-containment)
class BertClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dim=256):
        super(BertClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Instantiate the model with new weights
model_bs32_lr1e5 = BertClassifier(input_dim, num_classes)

# Define loss function and optimizer with LR = 1e-5
criterion_bs32_lr1e5 = nn.CrossEntropyLoss()
optimizer_bs32_lr1e5 = optim.Adam(model_bs32_lr1e5.parameters(), lr=0.00001) # Learning rate changed to 1e-5

# Create DataLoaders with batch_size = 32
batch_size_bs32_lr1e5 = 32 # Batch size changed to 32
train_dataset_bs32_lr1e5 = TensorDataset(X_train_tensor_bs32_lr1e5, y_train_tensor_bs32_lr1e5)
train_loader_bs32_lr1e5 = DataLoader(train_dataset_bs32_lr1e5, batch_size=batch_size_bs32_lr1e5, shuffle=True)

val_dataset_bs32_lr1e5 = TensorDataset(X_val_tensor_bs32_lr1e5, y_val_tensor_bs32_lr1e5)
val_loader_bs32_lr1e5 = DataLoader(val_dataset_bs32_lr1e5, batch_size=batch_size_bs32_lr1e5, shuffle=False)

test_dataset_bs32_lr1e5 = TensorDataset(X_test_tensor_bs32_lr1e5, y_test_tensor_bs32_lr1e5)
test_loader_bs32_lr1e5 = DataLoader(test_dataset_bs32_lr1e5, batch_size=batch_size_bs32_lr1e5, shuffle=False)

In [ ]:
# 3. Training loop with Validation
num_epochs = 10
print("Starting fine-tuning with Batch Size 32, Learning Rate 1e-5...")

for epoch in range(num_epochs):
    model_bs32_lr1e5.train() # Set model to training mode
    total_train_loss = 0
    for batch_X, batch_y in train_loader_bs32_lr1e5:
        optimizer_bs32_lr1e5.zero_grad() # Clear gradients
        outputs = model_bs32_lr1e5(batch_X) # Forward pass
        loss = criterion_bs32_lr1e5(outputs, batch_y) # Compute loss
        loss.backward() # Backward pass
        optimizer_bs32_lr1e5.step() # Update weights
        total_train_loss += loss.item()

    # Validation step
    model_bs32_lr1e5.eval() # Set model to evaluation mode
    total_val_loss = 0
    val_correct = 0
    total_val_samples = 0
    with torch.no_grad(): # Disable gradient calculation during evaluation
        for batch_X_val, batch_y_val in val_loader_bs32_lr1e5:
            outputs_val = model_bs32_lr1e5(batch_X_val)
            loss_val = criterion_bs32_lr1e5(outputs_val, batch_y_val)
            total_val_loss += loss_val.item()
            _, predicted_val = torch.max(outputs_val.data, 1) # Get the class with the highest probability
            total_val_samples += batch_y_val.size(0)
            val_correct += (predicted_val == batch_y_val).sum().item()

    val_accuracy = val_correct / total_val_samples

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {total_train_loss/len(train_loader_bs32_lr1e5):.4f}, Val Loss: {total_val_loss/len(val_loader_bs32_lr1e5):.4f}, Val Accuracy: {val_accuracy:.4f}")

print("Fine-tuning complete for Batch Size 32, Learning Rate 1e-5.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Data extracted from the output of the training cell (4EgZogMPzbfi)
epochs_bs32_lr1e5 = range(1, 11)
train_losses_bs32_lr1e5 = [0.9403, 0.8322, 0.8058, 0.7900, 0.7721, 0.7610, 0.7543, 0.7418, 0.7359, 0.7289]
val_losses_bs32_lr1e5 = [0.8554, 0.8187, 0.7996, 0.7861, 0.7748, 0.7667, 0.7590, 0.7522, 0.7464, 0.7407]
val_accuracies_bs32_lr1e5 = [0.6443, 0.6681, 0.6709, 0.6835, 0.6835, 0.6863, 0.6863, 0.6877, 0.6947, 0.6989]

plt.figure(figsize=(12, 5))

# Plotting Loss
plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
plt.plot(epochs_bs32_lr1e5, train_losses_bs32_lr1e5, label='Training Loss')
plt.plot(epochs_bs32_lr1e5, val_losses_bs32_lr1e5, label='Validation Loss')
plt.title('Training and Validation Loss per Epoch (BS 32, LR 1e-5)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plotting Accuracy
plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
plt.plot(epochs_bs32_lr1e5, val_accuracies_bs32_lr1e5, label='Validation Accuracy', color='green')
plt.title('Validation Accuracy per Epoch (BS 32, LR 1e-5)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# 4. Evaluation
model_bs32_lr1e5.eval() # Set model to evaluation mode
all_preds_bs32_lr1e5 = []
all_labels_bs32_lr1e5 = []

with torch.no_grad(): # Disable gradient calculation during evaluation
    for batch_X, batch_y in test_loader_bs32_lr1e5:
        outputs = model_bs32_lr1e5(batch_X)
        _, predicted = torch.max(outputs.data, 1) # Get the class with the highest probability
        all_preds_bs32_lr1e5.extend(predicted.cpu().numpy())
        all_labels_bs32_lr1e5.extend(batch_y.cpu().numpy())

# Convert numerical predictions back to original labels for classification report
y_pred_bs32_lr1e5 = label_encoder_bs32_lr1e5.inverse_transform(all_preds_bs32_lr1e5)
y_test_original_bs32_lr1e5 = label_encoder_bs32_lr1e5.inverse_transform(all_labels_bs32_lr1e5)

print("\nEvaluation Results (Batch Size 32, Learning Rate 1e-5):")
print(f"Accuracy: {accuracy_score(y_test_original_bs32_lr1e5, y_pred_bs32_lr1e5):.4f}")
print(f"Precision: {precision_score(y_test_original_bs32_lr1e5, y_pred_bs32_lr1e5, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test_original_bs32_lr1e5, y_pred_bs32_lr1e5, average='weighted'):.4f}")
print(f"F1-Score: {f1_score(y_test_original_bs32_lr1e5, y_pred_bs32_lr1e5, average='weighted'):.4f}")

print("\nClassification Report:")
print(classification_report(y_test_original_bs32_lr1e5, y_pred_bs32_lr1e5, target_names=label_encoder_bs32_lr1e5.classes_, digits=4))

In [ ]:
# Confusion Matrix for Batch Size 32, Learning Rate 1e-5
class_labels_bs32_lr1e5 = label_encoder_bs32_lr1e5.inverse_transform(np.arange(len(label_encoder_bs32_lr1e5.classes_)))
cm_bs32_lr1e5 = confusion_matrix(y_test_original_bs32_lr1e5, y_pred_bs32_lr1e5, labels=class_labels_bs32_lr1e5)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_bs32_lr1e5, annot=True, fmt='d', cmap='Oranges', xticklabels=class_labels_bs32_lr1e5, yticklabels=class_labels_bs32_lr1e5)
plt.title('Confusion Matrix for Batch Size 32, Learning Rate 1e-5')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

### **Batch Size 32 Learning Rate 1e-5 with Oversampling**

In [ ]:
# Re-encode labels for consistency, though label_encoder_balanced is already defined
# This ensures the new blocks use the correct encoding if run independently
label_encoder_balanced_bs32_lr1e5_oversampled = LabelEncoder()
y_train_encoded_balanced_bs32_lr1e5_oversampled = label_encoder_balanced_bs32_lr1e5_oversampled.fit_transform(y_train)
y_val_encoded_balanced_bs32_lr1e5_oversampled = label_encoder_balanced_bs32_lr1e5_oversampled.transform(y_val)
y_test_encoded_balanced_bs32_lr1e5_oversampled = label_encoder_balanced_bs32_lr1e5_oversampled.transform(y_test)

# Convert resampled data to PyTorch tensors
X_train_oversampled_tensor_bs32_lr1e5 = torch.tensor(X_train_resampled_embeddings.values, dtype=torch.float32)
y_train_oversampled_tensor_bs32_lr1e5 = torch.tensor(label_encoder_balanced_bs32_lr1e5_oversampled.transform(y_train_resampled), dtype=torch.long)

# Create DataLoader for oversampled training data
train_dataset_oversampled_bs32_lr1e5 = TensorDataset(X_train_oversampled_tensor_bs32_lr1e5, y_train_oversampled_tensor_bs32_lr1e5)
train_loader_oversampled_bs32_lr1e5 = DataLoader(train_dataset_oversampled_bs32_lr1e5, batch_size=32, shuffle=True)

# Re-instantiate the model for oversampling (to ensure fresh weights)
model_oversampled_bs32_lr1e5 = BertClassifier(input_dim, num_classes)
criterion_oversampled_bs32_lr1e5 = nn.CrossEntropyLoss()
optimizer_oversampled_bs32_lr1e5 = optim.Adam(model_oversampled_bs32_lr1e5.parameters(), lr=0.00001)

print("\n--- Training Model with Oversampled Data (Batch Size 32, Learning Rate 1e-5) ---")
num_epochs = 10
for epoch in range(num_epochs):
    model_oversampled_bs32_lr1e5.train()
    total_train_loss = 0
    for batch_X, batch_y in train_loader_oversampled_bs32_lr1e5:
        optimizer_oversampled_bs32_lr1e5.zero_grad()
        outputs = model_oversampled_bs32_lr1e5(batch_X)
        loss = criterion_oversampled_bs32_lr1e5(outputs, batch_y)
        loss.backward()
        optimizer_oversampled_bs32_lr1e5.step()
        total_train_loss += loss.item()

    model_oversampled_bs32_lr1e5.eval()
    total_val_loss = 0
    val_correct = 0
    total_val_samples = 0
    with torch.no_grad():
        for batch_X_val, batch_y_val in val_loader:
            outputs_val = model_oversampled_bs32_lr1e5(batch_X_val)
            loss_val = criterion_oversampled_bs32_lr1e5(outputs_val, batch_y_val)
            total_val_loss += loss_val.item()
            _, predicted_val = torch.max(outputs_val.data, 1)
            total_val_samples += batch_y_val.size(0)
            val_correct += (predicted_val == batch_y_val).sum().item()
    val_accuracy = val_correct / total_val_samples
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss (Oversampled): {total_train_loss/len(train_loader_oversampled_bs32_lr1e5):.4f}, Val Loss: {total_val_loss/len(val_loader):.4f}, Val Accuracy: {val_accuracy:.4f}")

print("Oversampling model training complete (Batch Size 32, Learning Rate 1e-5).")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Data extracted from the output of the oversampling training cell (s0KnhfCazkZT)
epochs_oversampled_bs32_lr1e5 = range(1, 11)
train_losses_oversampled_bs32_lr1e5 = [1.0315, 0.9474, 0.9160, 0.8931, 0.8792, 0.8612, 0.8480, 0.8337, 0.8246, 0.8134]
val_losses_oversampled_bs32_lr1e5 = [0.9827, 0.9356, 0.9253, 0.8949, 0.8847, 0.8511, 0.8554, 0.8516, 0.8335, 0.8424]
val_accuracies_oversampled_bs32_lr1e5 = [0.5504, 0.5728, 0.5714, 0.6008, 0.6022, 0.6317, 0.6232, 0.6246, 0.6345, 0.6289]

plt.figure(figsize=(12, 5))

# Plotting Loss
plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
plt.plot(epochs_oversampled_bs32_lr1e5, train_losses_oversampled_bs32_lr1e5, label='Training Loss (Oversampled)')
plt.plot(epochs_oversampled_bs32_lr1e5, val_losses_oversampled_bs32_lr1e5, label='Validation Loss')
plt.title('Training and Validation Loss per Epoch (Oversampling BS 32, LR 1e-5)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plotting Accuracy
plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
plt.plot(epochs_oversampled_bs32_lr1e5, val_accuracies_oversampled_bs32_lr1e5, label='Validation Accuracy', color='green')
plt.title('Validation Accuracy per Epoch (Oversampling BS 32, LR 1e-5)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# --- Evaluation with Oversampling Model (Batch Size 32, Learning Rate 1e-5) ---
model_oversampled_bs32_lr1e5.eval()
all_preds_oversampled_bs32_lr1e5 = []
all_labels_oversampled_bs32_lr1e5 = []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        outputs = model_oversampled_bs32_lr1e5(batch_X)
        _, predicted = torch.max(outputs.data, 1)
        all_preds_oversampled_bs32_lr1e5.extend(predicted.cpu().numpy())
        all_labels_oversampled_bs32_lr1e5.extend(batch_y.cpu().numpy())

y_pred_oversampled_bs32_lr1e5 = label_encoder_balanced_bs32_lr1e5_oversampled.inverse_transform(all_preds_oversampled_bs32_lr1e5)
y_test_original_oversampled_bs32_lr1e5 = label_encoder_balanced_bs32_lr1e5_oversampled.inverse_transform(all_labels_oversampled_bs32_lr1e5)

print("\nEvaluation Results (Oversampling (BS 32, LR 1e-5)):")
print(f"Accuracy: {accuracy_score(y_test_original_oversampled_bs32_lr1e5, y_pred_oversampled_bs32_lr1e5):.4f}")
print(f"Precision: {precision_score(y_test_original_oversampled_bs32_lr1e5, y_pred_oversampled_bs32_lr1e5, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test_original_oversampled_bs32_lr1e5, y_pred_oversampled_bs32_lr1e5, average='weighted'):.4f}")
print(f"F1-Score: {f1_score(y_test_original_oversampled_bs32_lr1e5, y_pred_oversampled_bs32_lr1e5, average='weighted'):.4f}")
print("Classification Report:")
print(classification_report(y_test_original_oversampled_bs32_lr1e5, y_pred_oversampled_bs32_lr1e5, target_names=label_encoder_balanced_bs32_lr1e5_oversampled.classes_, digits=4))

# Confusion Matrix for Oversampling
class_labels_balanced_bs32_lr1e5_oversampled = label_encoder_balanced_bs32_lr1e5_oversampled.inverse_transform(np.arange(len(label_encoder_balanced_bs32_lr1e5_oversampled.classes_)))
cm_oversampled_bs32_lr1e5 = confusion_matrix(y_test_original_oversampled_bs32_lr1e5, y_pred_oversampled_bs32_lr1e5, labels=class_labels_balanced_bs32_lr1e5_oversampled)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_oversampled_bs32_lr1e5, annot=True, fmt='d', cmap='Greens', xticklabels=class_labels_balanced_bs32_lr1e5_oversampled, yticklabels=class_labels_balanced_bs32_lr1e5_oversampled)
plt.title('Confusion Matrix (Oversampling (Batch Size 32, Learning Rate 1e-5))')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

### **Batch Size 32 Learning Rate 1e-5 with Undersampling**

In [ ]:
# Convert undersampled data to PyTorch tensors
X_train_undersampled_tensor_bs32_lr1e5 = torch.tensor(X_train_undersampled_embeddings.values, dtype=torch.float32)
y_train_undersampled_tensor_bs32_lr1e5 = torch.tensor(label_encoder_balanced_bs32_lr1e5_oversampled.transform(y_train_undersampled), dtype=torch.long)

# Create DataLoader for undersampled training data
train_dataset_undersampled_bs32_lr1e5 = TensorDataset(X_train_undersampled_tensor_bs32_lr1e5, y_train_undersampled_tensor_bs32_lr1e5)
train_loader_undersampled_bs32_lr1e5 = DataLoader(train_dataset_undersampled_bs32_lr1e5, batch_size=32, shuffle=True)

# Re-instantiate the model for undersampling (to ensure fresh weights)
model_undersampled_bs32_lr1e5 = BertClassifier(input_dim, num_classes)
criterion_undersampled_bs32_lr1e5 = nn.CrossEntropyLoss()
optimizer_undersampled_bs32_lr1e5 = optim.Adam(model_undersampled_bs32_lr1e5.parameters(), lr=0.00001)

print("\n--- Training Model with Undersampled Data (Batch Size 32, Learning Rate 1e-5) ---")
num_epochs = 10
for epoch in range(num_epochs):
    model_undersampled_bs32_lr1e5.train()
    total_train_loss = 0
    for batch_X, batch_y in train_loader_undersampled_bs32_lr1e5:
        optimizer_undersampled_bs32_lr1e5.zero_grad()
        outputs = model_undersampled_bs32_lr1e5(batch_X)
        loss = criterion_undersampled_bs32_lr1e5(outputs, batch_y)
        loss.backward()
        optimizer_undersampled_bs32_lr1e5.step()
        total_train_loss += loss.item()

    model_undersampled_bs32_lr1e5.eval()
    total_val_loss = 0
    val_correct = 0
    total_val_samples = 0
    with torch.no_grad():
        for batch_X_val, batch_y_val in val_loader:
            outputs_val = model_undersampled_bs32_lr1e5(batch_X_val)
            loss_val = criterion_undersampled_bs32_lr1e5(outputs_val, batch_y_val)
            total_val_loss += loss_val.item()
            _, predicted_val = torch.max(outputs_val.data, 1)
            total_val_samples += batch_y_val.size(0)
            val_correct += (predicted_val == batch_y_val).sum().item()
    val_accuracy = val_correct / total_val_samples
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss (Undersampled): {total_train_loss/len(train_loader_undersampled_bs32_lr1e5):.4f}, Val Loss: {total_val_loss/len(val_loader):.4f}, Val Accuracy: {val_accuracy:.4f}")

print("Undersampling model training complete (Batch Size 32, Learning Rate 1e-5).")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Data extracted from the output of the undersampling training cell (n3d8grllzv56)
epochs_undersampled_bs32_lr1e5 = range(1, 11)
train_losses_undersampled_bs32_lr1e5 = [1.0821, 1.0172, 0.9896, 0.9682, 0.9556, 0.9446, 0.9279, 0.9263, 0.9230, 0.9123]
val_losses_undersampled_bs32_lr1e5 = [1.0303, 0.9875, 0.9804, 0.9576, 0.9469, 0.9358, 0.9295, 0.9303, 0.9193, 0.9139]
val_accuracies_undersampled_bs32_lr1e5 = [0.5616, 0.5756, 0.5476, 0.5560, 0.5616, 0.5742, 0.5728, 0.5742, 0.5826, 0.5826]

plt.figure(figsize=(12, 5))

# Plotting Loss
plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
plt.plot(epochs_undersampled_bs32_lr1e5, train_losses_undersampled_bs32_lr1e5, label='Training Loss (Undersampled)')
plt.plot(epochs_undersampled_bs32_lr1e5, val_losses_undersampled_bs32_lr1e5, label='Validation Loss')
plt.title('Training and Validation Loss per Epoch (Undersampling BS 32, LR 1e-5)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plotting Accuracy
plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
plt.plot(epochs_undersampled_bs32_lr1e5, val_accuracies_undersampled_bs32_lr1e5, label='Validation Accuracy', color='green')
plt.title('Validation Accuracy per Epoch (Undersampling BS 32, LR 1e-5)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# --- Evaluation with Undersampling Model (Batch Size 32, Learning Rate 1e-5) ---
model_undersampled_bs32_lr1e5.eval()
all_preds_undersampled_bs32_lr1e5 = []
all_labels_undersampled_bs32_lr1e5 = []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        outputs = model_undersampled_bs32_lr1e5(batch_X)
        _, predicted = torch.max(outputs.data, 1)
        all_preds_undersampled_bs32_lr1e5.extend(predicted.cpu().numpy())
        all_labels_undersampled_bs32_lr1e5.extend(batch_y.cpu().numpy())

y_pred_undersampled_bs32_lr1e5 = label_encoder_balanced_bs32_lr1e5_oversampled.inverse_transform(all_preds_undersampled_bs32_lr1e5)
y_test_original_undersampled_bs32_lr1e5 = label_encoder_balanced_bs32_lr1e5_oversampled.inverse_transform(all_labels_undersampled_bs32_lr1e5)

print("\nEvaluation Results (Undersampling (BS 32, LR 1e-5)):")
print(f"Accuracy: {accuracy_score(y_test_original_undersampled_bs32_lr1e5, y_pred_undersampled_bs32_lr1e5):.4f}")
print(f"Precision: {precision_score(y_test_original_undersampled_bs32_lr1e5, y_pred_undersampled_bs32_lr1e5, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test_original_undersampled_bs32_lr1e5, y_pred_undersampled_bs32_lr1e5, average='weighted'):.4f}")
print(f"F1-Score: {f1_score(y_test_original_undersampled_bs32_lr1e5, y_pred_undersampled_bs32_lr1e5, average='weighted'):.4f}")
print("Classification Report:")
print(classification_report(y_test_original_undersampled_bs32_lr1e5, y_pred_undersampled_bs32_lr1e5, target_names=label_encoder_balanced_bs32_lr1e5_oversampled.classes_, digits=4))

# Confusion Matrix for Undersampling
cm_undersampled_bs32_lr1e5 = confusion_matrix(y_test_original_undersampled_bs32_lr1e5, y_pred_undersampled_bs32_lr1e5, labels=class_labels_balanced_bs32_lr1e5_oversampled)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_undersampled_bs32_lr1e5, annot=True, fmt='d', cmap='Oranges', xticklabels=class_labels_balanced_bs32_lr1e5_oversampled, yticklabels=class_labels_balanced_bs32_lr1e5_oversampled)
plt.title('Confusion Matrix (Undersampling (Batch Size 32, Learning Rate 1e-5))')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Data collected from previous model evaluations
results = {
    'Model Configuration': [
        'Baseline (BS 16, LR 2e-5)',
        'Oversampling (BS 16, LR 2e-5)',
        'Undersampling (BS 16, LR 2e-5)',
        '(BS 32, LR 2e-5)',
        'Oversampling (BS 32, LR 2e-5)',
        'Undersampling (BS 32, LR 2e-5)',
        '(BS 16, LR 1e-5)',
        'Oversampling (BS 16, LR 1e-5)',
        'Undersampling (BS 16, LR 1e-5)',
        '(BS 32, LR 1e-5)',
        'Oversampling (BS 32, LR 1e-5)',
        'Undersampling (BS 32, LR 1e-5)'
    ],
    'Accuracy': [
        0.7479, 0.7143, 0.6625,
        0.7339, 0.7241, 0.6303,
        0.7269, 0.7017, 0.5784,
        0.7045, 0.6597, 0.6036
    ],
    'Precision': [
        0.7510, 0.7559, 0.6908,
        0.7271, 0.7489, 0.6832,
        0.7373, 0.7212, 0.6747,
        0.7588, 0.7069, 0.6674
    ],
    'Recall': [
        0.7479, 0.7143, 0.6625,
        0.7339, 0.7241, 0.6303,
        0.7269, 0.7017, 0.5784,
        0.7045, 0.6597, 0.6036
    ],
    'F1-Score': [
        0.7044, 0.7273, 0.6698,
        0.6901, 0.7330, 0.6451,
        0.6748, 0.7091, 0.5987,
        0.6260, 0.6733, 0.6199
    ]
}

df_results = pd.DataFrame(results)

print("\n--- Model Comparison Table ---")
display(df_results)

In [ ]:
# Plotting Accuracy for comparison
plt.figure(figsize=(14, 7))
sns.barplot(x='Accuracy', y='Model Configuration', data=df_results.sort_values(by='Accuracy', ascending=False), palette='viridis')
plt.title('Comparison of Accuracy Across Different Model Configurations')
plt.xlabel('Accuracy (Weighted Average)')
plt.ylabel('Model Configuration')
plt.xlim(0.5, 0.75) # Adjust x-axis limit for better visualization of differences
plt.tight_layout()
plt.show()

In [ ]:
df_melted = df_results.melt(id_vars='Model Configuration', var_name='Metric', value_name='Score')

plt.figure(figsize=(16, 8))
sns.barplot(x='Model Configuration', y='Score', hue='Metric', data=df_melted, palette='tab10')
plt.title('All Evaluation Metrics Across Different Model Configurations')
plt.xlabel('Model Configuration')
plt.ylabel('Score')
plt.xticks(rotation=45, ha='right')
plt.ylim(0.5, 0.8) # Adjust y-axis limit for better visualization
plt.tight_layout()
plt.show()